In [21]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
import imageio

In [22]:
def carregar_csv(caminho):
    df = pd.read_csv(caminho, header=None, sep='\t')

    df[0] = df[0].astype(float)
    df[1] = df[1].astype(float)
    df[2] = df[2].astype(int)

    dados = df[[0, 1]].values
    rotulos = df[2].values - 1

    return dados, rotulos

In [23]:
def gerar_dados():
    dados = []
    rotulos = []

    centros = [(-20, -20), (20, 20), (-20, 20), (20, -20)]

    for i, (cx, cy) in enumerate(centros):
        for _ in range(100):
            x = np.random.normal(cx, 2)
            y = np.random.normal(cy, 2)
            dados.append([x, y])
            rotulos.append(i)

    return np.array(dados), np.array(rotulos)

In [24]:
def normalizar(dados):
    min_vals = dados.min(axis=0)
    max_vals = dados.max(axis=0)
    return (dados - min_vals) / (max_vals - min_vals)

In [25]:
def distribuir_dados(grid, dados, rotulos):
    linhas, colunas = grid.shape
    indices = np.random.choice(linhas * colunas, len(dados), replace=False)
    xs, ys = np.unravel_index(indices, (linhas, colunas))

    for i in range(len(dados)):
        grid[xs[i], ys[i]] = (dados[i], rotulos[i])

In [26]:
def distancia(a, b):
    return np.linalg.norm(a - b)


def similaridade(item_dados, vizinho_dados, alpha):
    distancia_item = distancia(item_dados, vizinho_dados)
    return max(0.0, 1 - distancia_item / alpha)


In [27]:
def visualizar(grid):
    plt.figure(figsize=(6,6))
    cores = ['red', 'blue', 'green', 'purple', 'yellow', 'cyan', 'magenta', 'orange', 
             'lime', 'navy', 'pink', 'olive', 'teal', 'plum', 'salmon']

    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            if grid[i, j] is not None:
                _, label = grid[i, j]
                plt.scatter(i, j, color=cores[label % len(cores)], s=10)

    plt.title("Clusters formados")
    plt.show()

In [28]:
def renderizar_grid(grid, scale=6):
    img = np.zeros((grid.shape[0], grid.shape[1], 3), dtype=np.uint8)

    cores = [
        [255, 0, 0],      # vermelho
        [0, 0, 255],      # azul
        [0, 255, 0],      # verde
        [128, 0, 128],    # roxo
        [255, 255, 0],    # amarelo
        [0, 255, 255],    # ciano
        [255, 0, 255],    # magenta
        [255, 128, 0],    # laranja
        [128, 255, 0],    # verde-lima
        [0, 128, 255],    # azul-claro
        [255, 0, 128],    # rosa
        [128, 128, 0],    # olive
        [0, 128, 128],    # teal
        [128, 0, 255],    # roxo-claro
        [255, 128, 128]   # rosa-claro
    ]

    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            if grid[i, j] is not None:
                _, label = grid[i, j]
                img[i, j] = cores[label % len(cores)]

    # aumenta resolução
    img = np.kron(img, np.ones((scale, scale, 1)))

    return img.astype(np.uint8)

In [29]:
class Ant:
    def __init__(self, x, y, r=1, k1=0.3, k2=0.6, alpha=0.1):

        self.x = x
        self.y = y
        self.r = r

        self.carregando = False 

        self.k1 = k1
        self.k2 = k2

        self.alpha = alpha

        self.item = None
        
    def move(self, max_x, max_y):
        direcoes = [(0, 1), (0, -1), (1, 0), (-1, 0)]
        dx, dy = random.choice(direcoes)
        
        self.x = (self.x + dx) % max_x
        self.y = (self.y + dy) % max_y

    def get_density(self, matrix):
        linhas, colunas = matrix.shape
        item_dados = self.item[0] if self.carregando else matrix[self.x, self.y][0]

        soma = 0
        total_celulas = (2 * self.r + 1) ** 2
        if not self.carregando:
            total_celulas -= 1

        for i in range(self.x - self.r, self.x + self.r + 1):
            for j in range(self.y - self.r, self.y + self.r + 1):
                ii = i % linhas
                jj = j % colunas

                if not self.carregando and ii == self.x and jj == self.y:
                    continue

                if matrix[ii, jj] is not None:
                    vizinho_dados, _ = matrix[ii, jj]
                    soma += similaridade(item_dados, vizinho_dados, self.alpha)

        return soma / total_celulas if total_celulas > 0 else 0

    def decidir_acao(self, matrix):
        item_na_celula = matrix[self.x, self.y]

        # PEGAR
        if not self.carregando:
            if item_na_celula is not None:
                f = self.get_density(matrix)
                p_pegar = (self.k1 / (self.k1 + f)) ** 2

                if random.random() < p_pegar:
                    self.item = item_na_celula
                    matrix[self.x, self.y] = None
                    self.carregando = True
        #LARGAR
        else:
            if item_na_celula is None:
                f = self.get_density(matrix)
                p_largar = (f / (self.k2 + f)) ** 2

                if random.random() < p_largar:
                    matrix[self.x, self.y] = self.item
                    self.item = None
                    self.carregando = False


In [ ]:
LARGURA = 64
ALTURA = 64
NUM_FORMIGAS = 15  # 15 formigas vivas
MAX_INTERACOES = 3000000  # Reduzido para 5 milhões (mais realista)

dados, rotulos = carregar_csv("Base15.csv")
dados = normalizar(dados)  # Dados normalizados

# criar grid
grid = np.empty((LARGURA, ALTURA), dtype=object)
grid[:] = None

distribuir_dados(grid, dados, rotulos)

# criar formigas com parâmetros específicos para 15 grupos
formigas = [
    Ant(random.randint(0, LARGURA-1),
        random.randint(0, ALTURA-1),
        r=1, k1=0.9, k2=0.05, alpha=0.11)  # α = 0.11, k1 = 0.9, k2 = 0.05
    for _ in range(NUM_FORMIGAS)
]

print("Simulação iniciando...")

frame_inicial = renderizar_grid(grid)
imageio.imwrite("estado_inicial5.png", frame_inicial)

with imageio.get_writer("ant_clustering5.gif", mode="I", fps=5) as writer:

    for it in range(MAX_INTERACOES):
        random.shuffle(formigas)

        for ant in formigas:
            ant.move(LARGURA, ALTURA)
            ant.decidir_acao(grid)

        if it % 100000 == 0:  # Frames a cada 100k iterações
            print(f"Frame {it}")
            frame = renderizar_grid(grid)
            writer.append_data(frame)

frame_final = renderizar_grid(grid)
imageio.imwrite("estado_final5.png", frame_final)

print("Simulação finalizada!")

Simulação iniciando...
Frame 0
Frame 100000
Frame 200000
Frame 300000
Frame 400000
Frame 500000
Frame 600000
Frame 700000
Frame 800000
Frame 900000
Frame 1000000
Frame 1100000
Frame 1200000
Frame 1300000
Frame 1400000
Frame 1500000
Frame 1600000
Frame 1700000
Frame 1800000
Frame 1900000
Simulação finalizada!
